# Lab: Semantic Segmentation with U-Net for Road Extraction

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Intended Learning Outcomes (ILOs)
Upon successful completion of this lab, students will be able to:
* **Implement** a U-Net fully convolutional neural network from scratch using PyTorch to understand the mechanics of encoder-decoder architectures and skip connections.
* **Execute** a complete semantic segmentation pipeline to extract binary road networks from high-resolution, multi-modal data.
* **Design and conduct** a comparative empirical experiment evaluating the impact of distinct input modalities (optical satellite imagery vs. rendered maps vs. concatenated features) on model convergence and feature extraction.
* **Quantify** segmentation accuracy using industry-standard metrics (Jaccard Index/IoU and Dice Score) and systematically serialize experimental findings using Pydantic schemas.
* **Critically analyze** model failure modes by generating and interpreting side-by-side qualitative visual comparisons of predicted masks against ground-truth data.

> **Important Note on Performance Expectations:** > Your model's accuracy (IoU/Dice Score) should be acceptable and clearly demonstrate that learning and convergence occurred over the epochs. However, it does not need to achieve state-of-the-art or exceptionally high scores. The primary grading focus is on the correct implementation of the U-Net architecture's forward pass, the training loop mechanics, and your analytical conclusions.


In [ ]:
import os
import json
from enum import Enum
from typing import List, Dict, Any
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from pydantic import BaseModel
import warnings

warnings.filterwarnings('ignore')


### Experimental Setup: Input Modalities
To evaluate whether the network extracts roads more effectively using raw optical satellite imagery, rendered Google Maps data, or a combined approach, an Enum is defined to control the data loader's behavior. You will choose one of these modalities for your final experimental trial.

In [ ]:
class InputModality(Enum):
    SATELLITE_ONLY = 'satellite'   # 3-channel input from the 'images' folder
    MAP_ONLY = 'map'               # 3-channel input from the 'actual' folder
    BOTH_CONCATENATED = 'both'     # 6-channel input (concatenating both images)

# You will use these Enum values when instantiating your DataLoaders for the three experiments below.


### Dataset Overview and Directory Structure
The required dataset for this lab is the **Satellite-Googlemaps-Masks** dataset, available on Kaggle:
[https://www.kaggle.com/datasets/arka47/satellitegooglemapsmasks](https://www.kaggle.com/datasets/arka47/satellitegooglemapsmasks)

The downloaded dataset contains two main folders: `train` and `val`. 
* **Test Set:** You must strictly treat the original Kaggle `val` folder as your **Test Set** to evaluate your final model.
* **Train/Validation Split:** You must dynamically split the Kaggle `train` folder into your own training and validation subsets to monitor convergence and prevent overfitting.

Inside both the `train` and `val` folders, you will find three subdirectories containing perfectly aligned 600x600 images:
1. `images/`: The raw optical satellite imagery.
2. `actual/`: The rendered Google Maps representation.
3. `label/`: The binary ground truth mask highlighting the road networks.

Depending on the `InputModality` chosen in the previous cell, your custom Dataset class must load either the satellite image (3 channels), the map image (3 channels), or stack them together (6 channels). All images and masks must be resized to **512x512 pixels** using `torchvision.transforms` to ensure dimensional compatibility with the U-Net architecture. Masks must be converted to strict binary tensors (values of 0.0 or 1.0).


In [ ]:
class RoadSegmentationDataset(Dataset):
    def __init__(self, root_dir, modality: InputModality, transform=None):
        """
        Args:
            root_dir (string): Directory with all the images (e.g., path to 'train' or 'val').
            modality (InputModality): SATELLITE_ONLY, MAP_ONLY, or BOTH_CONCATENATED.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.modality = modality
        self.transform = transform
        
        # TODO: Dynamically load the filenames from the subdirectories
        # Hint: Use os.listdir() on the 'label' directory to get the base filenames, 
        # since the filenames are identical across 'images', 'actual', and 'label'.
        self.filenames = [] 

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # TODO: Construct the full file paths based on the requested idx
        # TODO: Load the mask from the 'label/' folder
        # TODO: Load the input(s) from 'images/' and/or 'actual/' based on self.modality
        # TODO: Apply self.transform if it exists (Remember to resize to 512x512!)
        # TODO: Return a tuple of (input_tensor, mask_tensor)
        
        pass

# TODO: Define your transforms (resize to 512x512, convert to Tensor, and normalize)
# TODO: Instantiate your datasets for the three experiments (e.g., train_sat, train_map, train_both, and their validation equivalents)
# TODO: Create PyTorch DataLoaders for each dataset


### U-Net Architecture implementation


The U-Net architecture consists of a **contracting path** (encoder) to capture context and a symmetric **expansive path** (decoder) that enables precise localization. 

The `__init__` function has been scaffolded for you. Notice that the first convolutional layer accepts a variable `in_channels` parameter. This is crucial for our experiment: it will be `3` if you are using only satellite or only map images, but it will be `6` if you are concatenating both modalities.

**Your Task:** Implement the `forward(self, x)` pass. You must systematically track the spatial tensors through the network and use `torch.cat` to concatenate the skip connections from the encoder to the corresponding layers in the decoder.


In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()
        
        # Contracting Path (Encoder)
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(128, 256))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(256, 512))
        self.down4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(512, 1024))
        
        # Expansive Path (Decoder)
        self.up1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(1024, 512)
        
        self.up2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.conv3 = DoubleConv(256, 128)
        
        self.up4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv4 = DoubleConv(128, 64)
        
        # Final Output Layer
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        # TODO: Implement the forward pass.
        # Ensure you capture intermediate outputs from self.inc, self.down1, self.down2, self.down3
        # Use torch.cat((up_sampled_tensor, skip_connection_tensor), dim=1) during the upward pass.
        
        pass


### Three-Part Experimental Training
To rigorously evaluate the impact of different input features, you must train three separate U-Net models:
1. **Model A:** Trained exclusively on raw satellite imagery (3 channels).
2. **Model B:** Trained exclusively on rendered Google Maps (3 channels).
3. **Model C:** Trained on a concatenated input of both satellite and map images (6 channels).

**Important Logistical Note:** Training three distinct models can be computationally expensive. You are strongly encouraged to modularize your training loop into a reusable Python function. To ensure the lab remains manageable within reasonable compute constraints, limit your training process to a moderate number of epochs (e.g., 5 to 10 epochs per model).

Remember, the goal is to achieve acceptable, comparative convergence, not state-of-the-art absolute accuracy.


In [ ]:
# TODO: Define your loss function (e.g., nn.BCEWithLogitsLoss)

# TODO: Write a reusable training function to avoid code duplication
# def train_unet(model, train_loader, val_loader, criterion, optimizer, num_epochs):
#     ...
#     return best_model

# TODO: Instantiate your three models (Sat-only, Map-only, Combined)
# Ensure the `in_channels` parameter matches the input modality for each.

# TODO: Train all three models and keep track of their best validation state.


### Qualitative Visualization


Visual context is critical for understanding network failure modes. Select exactly **3 distinct instances** from your Test set. 

For each instance, plot a 1x6 grid displaying the images side-by-side in this exact order:
`[Satellite Image] | [Map Image] | [Ground Truth Mask] | [Pred: Sat Only] | [Pred: Map Only] | [Pred: Combined]`

You must programmatically save this entire Matplotlib figure (which should be a 3x6 grid overall) to disk as `sample_predictions.png` so it can be included in your submission archive.


In [ ]:
# TODO: Fetch 3 distinct sample tuples from the Test DataLoader.
# TODO: Run the inputs through their respective trained models to generate predicted masks.
# TODO: Use matplotlib.pyplot to create the 3x6 grid.
# TODO: Save the figure using plt.savefig('sample_predictions.png', bbox_inches='tight')


### Quantitative Evaluation & JSON Export
Evaluate your three models on the test set utilizing the Jaccard Index (Intersection over Union - IoU) and the Dice Score. 

The `pydantic` library is utilized below to ensure the structural integrity of your final submission. Populate the models with your experimental findings and your analytical conclusion comparing the theoretical and practical value of map rendering versus raw optical imagery for road extraction.


In [ ]:
class ExperimentMetrics(BaseModel):
    modality: str
    test_iou: float
    test_dice: float

class LabSubmission(BaseModel):
    student_ids: List[str]
    epochs_run_per_model: int
    metrics: List[ExperimentMetrics]
    analytical_conclusion: str

# TODO: Find or implement a function to calculate the Intersection over Union (IoU) and Dice Score based on your model's predictions.
# TODO: Populate the submission object with your actual IDs, calculated metrics, and text analysis.
submission = LabSubmission(
    student_ids=[9000000, 9000001], # Update with your actual student IDs
    epochs_run_per_model=5, # Update with your actual epoch count
    metrics=[
        ExperimentMetrics(modality="Satellite Only", test_iou=0.0, test_dice=0.0),
        ExperimentMetrics(modality="Map Only", test_iou=0.0, test_dice=0.0),
        ExperimentMetrics(modality="Combined", test_iou=0.0, test_dice=0.0)
    ],
    analytical_conclusion="Insert your comparative analysis here. Discuss why certain modalities performed better or worse based on your visual and quantitative findings."
)

# Serialize to JSON and save
output_json = submission.model_dump_json(indent=4)
with open("student_submission.json", "w") as f:
    f.write(output_json)
    
print("Successfully exported student_submission.json")


### Submission Protocol

The final deliverable **must be a zipped archive** submitted on the classroom. 

* The archive must contain:
    1. The executed Jupyter Notebook.
    2. The generated `student_submission.json` file.
    3. The saved `sample_predictions.png` image.
* The zipped file must be explicitly named utilizing the display name of the submitter exactly as it appears on Google Classroom.

---

### Grading Rubric (Total: 10 Marks)

* **Data Engineering (2 Marks):** Correctly implements the multi-modal `Dataset` class, accurately pairs the inputs, and applies the necessary 512x512 transformations and tensor conversions.
* **Architecture Implementation (3 Marks):** Successfully completes the U-Net `forward` pass with proper spatial tracking, skip connections, and adaptable input channels.
* **Training Execution (2 Marks):** Implements a functional, reusable training loop that successfully trains the three experimental models and tracks loss.
* **Evaluation & JSON Export (2 Marks):** Correctly computes Test set IoU/Dice metrics and successfully serializes the experiment and written analysis into the strictly formatted JSON payload.
* **Qualitative Output (1 Mark):** Successfully generates, formats, and saves the 3-sample comparative plot (`sample_predictions.png`).
